# FineVision subset inspection

Peek at example rows (images + multi-turn conversation) from each downloaded FineVision subset.

Expects parquets under `/local/<user>/tool-vision/data/FineVision/<subset>/train-*.parquet` — download them first with `python download_finevision.py`.

In [ ]:
import io
from pathlib import Path

import pyarrow.parquet as pq
from IPython.display import Markdown, display
from PIL import Image

DATA = Path("/local/<user>/tool-vision/data/FineVision")
HERE = Path.cwd()
SUBSETS_FILE = HERE / "subsets.txt"


def load_subsets_from_file() -> list[str]:
    return [
        line.strip()
        for line in SUBSETS_FILE.read_text().splitlines()
        if line.strip() and not line.startswith("#")
    ]


def available_subsets() -> list[str]:
    return sorted(p.name for p in DATA.iterdir() if p.is_dir())


def peek(subset: str, n: int = 3, max_img_px: int = 512) -> None:
    files = sorted((DATA / subset).glob("train-*.parquet"))
    assert files, f"no parquet files under {DATA / subset}"
    pf = pq.ParquetFile(files[0])
    total_rows = sum(pf.metadata.row_group(i).num_rows for i in range(pf.num_row_groups))
    batch = next(pf.iter_batches(batch_size=n))
    df = batch.to_pandas()
    display(
        Markdown(
            f"## `{subset}`\n\n"
            f"{len(files)} parquet file(s); first file has ~{total_rows:,} rows. "
            f"Showing {len(df)} row(s)."
        )
    )
    for i, row in df.iterrows():
        display(Markdown(f"### Row {i} \u2014 source: `{row.get('source', 'n/a')}`"))
        for j, img_d in enumerate(row["images"]):
            img = Image.open(io.BytesIO(img_d["bytes"]))
            if max(img.size) > max_img_px:
                img.thumbnail((max_img_px, max_img_px))
            display(Markdown(f"*image {j} \u2014 {img.size[0]}\u00d7{img.size[1]}*"))
            display(img)
        for k, turn in enumerate(row["texts"]):
            display(
                Markdown(
                    f"**turn {k} \u2014 user:** {turn['user']}\n\n"
                    f"**turn {k} \u2014 assistant:** {turn['assistant']}"
                )
            )


print(f"DATA: {DATA}")
print(f"available subsets on disk: {available_subsets()}")

## Peek one subset

Edit the name below. Use any directory printed above.

In [ ]:
peek("CoSyn_400k_chart", n=3)

## Loop over every subset in `subsets.txt`

In [ ]:
for s in load_subsets_from_file():
    try:
        peek(s, n=2)
    except AssertionError as e:
        display(Markdown(f"**skipping `{s}`**: {e}"))